# Your scraper has an accent

Two clients. Same machine, same address, same URL. One gets a block page and
the other gets the site.

The only difference between them is what their TLS handshake and HTTP/2
connection look like on the wire. No proxy, no JavaScript, no captcha solver.

In [ ]:
import html
import json
import re

import httpx
from IPython.display import HTML, display

from pybrws.client import get

PROBE = "https://tls.peet.ws/api/all"
BLOCKED = (403, 503)

# Bright enough for a projector, in either notebook theme.
GOOD, BAD, WARN, DIM = "#2ea043", "#f85149", "#d29922", "#8b949e"
CELL = "padding:7px 18px 7px 0;white-space:nowrap;text-align:left;"


def fetch(client, url):
    """Return (status, body) from either client, or (None, error text)."""
    try:
        if client == "pybrws":
            response = get(url, follow_redirects=5, timeout=30)
            return response.status, response.text()
        with httpx.Client(http2=True, follow_redirects=True, timeout=30) as http:
            response = http.get(url)
        return response.status_code, response.text
    except Exception as error:
        return None, "%s: %s" % (type(error).__name__, error)


def page_title(page):
    found = re.search(r"<title[^>]*>(.*?)</title>", page, re.S | re.I)
    return html.escape(found.group(1).strip()[:70]) if found else "(no title)"


def show(label, status, page, height=360):
    """Summarise a result, then preview it in a sandboxed frame.

    The summary is plain markup so it survives a renderer that strips
    iframes; the preview is a bonus where frames are allowed.
    """
    served = status is not None and status not in BLOCKED and status != 429
    tone = GOOD if served else BAD
    display(HTML(
        '<div style="font:14px/1.6 system-ui;padding:10px 14px;border-radius:6px;'
        'border:1px solid %s"><b>%s</b>'
        '<span style="color:%s;font-weight:700;margin-left:12px">%s</span>'
        '<span style="color:%s;margin-left:12px">%s bytes</span>'
        '<div style="color:%s;margin-top:4px">&lt;title&gt; %s</div></div>'
        % (tone, html.escape(label), tone,
           "HTTP %s" % status if status else "failed",
           DIM, format(len(page), ","), DIM, page_title(page))
    ))
    display(HTML(
        '<iframe sandbox="" loading="lazy" style="width:100%%;height:%dpx;'
        'border:1px solid %s;border-radius:6px;background:#fff;margin-top:6px" '
        'srcdoc="%s"></iframe>' % (height, DIM, html.escape(page, quote=True))
    ))


def compare(url, height=360):
    for client in ("httpx", "pybrws"):
        status, page = fetch(client, url)
        show("%s  %s" % (client, url), status, page, height)


def table(header, rows):
    head = "".join('<th style="%s">%s</th>' % (CELL, h) for h in header)
    body = "".join("<tr>%s</tr>" % "".join(r) for r in rows)
    display(HTML(
        '<div style="overflow-x:auto"><table style="font:13px/1.4 ui-monospace,'
        'SFMono-Regular,Menlo,monospace;border-collapse:collapse">'
        '<tr style="border-bottom:1px solid %s;color:%s">%s</tr>%s</table></div>'
        % (DIM, DIM, head, body)
    ))


def td(text, colour=None, bold=False):
    return '<td style="%scolor:%s;%s">%s</td>' % (
        CELL, colour or "inherit", "font-weight:700" if bold else "", text)


def verdict(status):
    """Classify a status: served, blocked, or rate limited by our own traffic."""
    if status is None:
        return BAD, "&#10007; failed"
    if status == 429:
        return WARN, "&#8211; 429 rate limited"
    if status in BLOCKED:
        return BAD, "&#10007; %d" % status
    return GOOD, "&#10003; %d" % status


## 1. zillow.com

`httpx` is a mature HTTP/2 client over Python's standard TLS stack. `pybrws`
differs from it only in the bytes of its ClientHello, its HTTP/2 preamble and
its header order.

In [ ]:
compare("https://www.zillow.com/")

## 2. etsy.com, behind DataDome

A category page rather than the homepage. The strictest rules sit on the
endpoints most worth scraping: etsy refuses `/` and `/?q=` to this client too,
while serving `/c/jewelry`.

In [ ]:
compare("https://www.etsy.com/c/jewelry")

## 3. A wider sweep

Four protection vendors, measured the same way. The rows where this does not
work are included: a target both clients fail is not a failure of the
measurement, and leaving it out would be choosing the result.

`429` is rate limiting earned by a day of measuring, not a verdict on the
client, so it is marked apart from a block.

In [ ]:
TARGETS = [
    ("https://www.zillow.com/", "CloudFront"),
    ("https://www.etsy.com/c/jewelry", "DataDome"),
    ("https://www.yelp.com/", "DataDome"),
    ("https://www.homedepot.com/", "Akamai"),
    ("https://www.quora.com/", "Cloudflare"),
    ("https://www.etsy.com/", "DataDome, homepage"),
    ("https://www.g2.com/", "Cloudflare"),
    ("https://example.com/", "control, serves everyone"),
]



rows = []
for url, guard in TARGETS:
    cells = [td(html.escape(url)), td(guard, DIM)]
    for client in ("pybrws", "httpx"):
        colour, text = verdict(fetch(client, url)[0])
        cells.append(td(text, colour, bold=True))
    rows.append(cells)

table(["target", "protection", "pybrws", "httpx"], rows)

## 4. Why the baseline arm is not optional

A query string gets both sites to serve a request they otherwise refuse. It is
tempting to read that as the client working. Running the plain client beside it
says otherwise on one of the two.

The same parameter, five interleaved rounds each.

In [ ]:
PARAM_TARGETS = ["https://www.g2.com/?x=", "https://www.etsy.com/?x="]

rows = []
for url in PARAM_TARGETS:
    cells = [td(html.escape(url))]
    results = {}
    for client in ("pybrws", "httpx"):
        status = fetch(client, url)[0]
        results[client] = status
        colour, text = verdict(status)
        cells.append(td(text, colour, bold=True))
    served = lambda s: s is not None and s not in BLOCKED and s != 429
    if served(results["pybrws"]) and not served(results["httpx"]):
        reading, colour = "the fingerprint is what decides", GOOD
    elif served(results["pybrws"]) and served(results["httpx"]):
        reading, colour = "the query string alone does it, we add nothing", WARN
    else:
        reading, colour = "no difference", DIM
    cells.append(td(reading, colour))
    rows.append(cells)

table(["target", "pybrws", "httpx", "reading"], rows)

Without the second column, both rows look like a win. One of them is not.

Every claim in this project carries that arm, which is also why three findings
during its development were retracted rather than published.

## 5. What the server actually saw

`tls.peet.ws` reports back the fingerprint of whoever asked. The first row is
a real Firefox 157, captured by opening that URL in the browser.

In [ ]:
with open("fixtures/reference-firefox-157-peet.json", encoding="utf-8") as handle:
    reference = json.load(handle)

with httpx.Client(http2=True, timeout=30) as http:
    seen_baseline = http.get(PROBE).json()
seen_ours = json.loads(get(PROBE, timeout=30).body)

target_ja4 = reference["tls"]["ja4"]
target_h2 = reference["http2"]["akamai_fingerprint"]

rows = []
for index, (name, seen) in enumerate(
    [("Firefox 157", reference), ("pybrws", seen_ours), ("httpx", seen_baseline)]
):
    cells = [td(name, bold=True)]
    for value, target in ((seen["tls"]["ja4"], target_ja4),
                          (seen["http2"]["akamai_fingerprint"], target_h2)):
        if index == 0:
            cells.append(td(html.escape(value), DIM))
        else:
            match = value == target
            mark = "&#10003; " if match else "&#10007; "
            cells.append(td(mark + html.escape(value), GOOD if match else BAD))
    rows.append(cells)

table(["client", "JA4", "HTTP/2 fingerprint"], rows)

## 6. Does a session change the outcome?

Both etsy and g2 refuse one endpoint while serving another. A browser never
arrives at a page cold: it has been around the site, it carries cookies, and
it reuses the connection it already opened.

So does reaching a refused endpoint *after* a served one change anything?

Four arms. The first three separate what a session changes, since it carries
cookies and reuses a connection at once:

| arm | cookies | connection |
|---|---|---|
| **cold** | none | new |
| **cookies only** | carried | new, the pool is dropped in between |
| **warm** | carried | reused |

The fourth is the one that matters for architecture: hand those cookies to
the plain client and see whether they still work. If they travel, a browser
can earn a token once and anything cheap can spend it.

In [ ]:
from pybrws.session import Session

WARMUP, REFUSED = "https://www.etsy.com/c/jewelry", "https://www.etsy.com/"


def cold():
    return fetch("pybrws", REFUSED)[0]


def cookies_only():
    with Session() as session:
        session.get(WARMUP)
        session.close()      # keeps the jar, drops the pooled connection
        return session.get(REFUSED).status


def warm():
    with Session() as session:
        session.get(WARMUP)
        return session.get(REFUSED).status


def handed_to_httpx():
    """Earn the cookies here, spend them from the plain client."""
    with Session() as session:
        session.get(WARMUP)
        jar = {cookie.name: cookie.value for cookie in session.cookies}
    with httpx.Client(timeout=30, http2=True, follow_redirects=True,
                      cookies=jar) as http:
        return http.get(REFUSED).status_code, len(jar)


ARMS = [("pybrws, cold", cold), ("pybrws, cookies only", cookies_only),
        ("pybrws, warm", warm)]

rows = []
for label, run in ARMS:
    colour, text = verdict(run())
    rows.append([td(label), td(text, colour, bold=True), td("", DIM)])

status, carried = handed_to_httpx()
colour, text = verdict(status)
rows.append([td("httpx, given those cookies", BAD if colour is BAD else None),
             td(text, colour, bold=True),
             td("%d cookies carried over" % carried, DIM)])

table(["arm", "result", ""], rows)

One measurement is not a finding. These sites have already been seen to answer
the same request differently from one run to the next, so repeat anything
green here before believing it; `tools/ab_test.py` keeps a timestamped log.

Measured over four interleaved rounds, with no variance in any arm: the first
three come out `0/4`, `4/4`, `4/4`, and the fourth comes out `0/4`.

So the cookies do the work and the connection does not, and they are **not
transferable**. The same cookie values, sent by a client with a different
fingerprint, are refused every time. The token is bound to whoever earned it.

That rules out the cheapest architecture, where a browser collects a cookie
and anything at all spends it. What is left still works, but the client doing
the spending has to carry the fingerprint too, which is what this one is.

## 7. The part that is easy to oversell

The fingerprint is necessary and it is not sufficient. Crossing it against the
address it comes from, three rounds each:

| | pybrws | httpx |
|---|---|---|
| **home connection** | 3/3 served | 0/3 |
| **commercial ISP proxy** | 0/3 | 0/3 |

One cell of four passes. A proxy address sold as clean is itself a signal, and
here it scored worse than an ordinary domestic line.

And nothing here solves a JavaScript challenge. A site that issues one is out
of reach for any HTTP client, by construction.

## 8. A second browser, which is how you find out the first one taught you wrong

Everything above is Firefox. That is one implementation, and a profile system
with one implementation is an assumption wearing a costume.

Chrome breaks it, which is what it is for. Measured from six captures of its
ClientHello plus one each of its HTTP/2 and HTTP/1.1 requests, the two
browsers disagree about nearly everything -- including *whether the extension
order is a fact at all*. Firefox sends the same order every time. Chrome draws
a fresh one per connection, so a profile that stores an order would be
recording one sample as though it were a constant.

The cell below needs no network: it builds hellos and reads them back.

In [ ]:
from pybrws.emitter import build_client_hello
from pybrws.model import GREASE_VALUES
from pybrws.parser import parse_client_hello
from pybrws.profiles import chrome155, firefox157


def shape(profile, rounds=8):
    """What this profile looks like over several connections."""
    hellos = [parse_client_hello(build_client_hello("example.com", profile=profile))
              for _ in range(rounds)]
    orders = {tuple("G" if t in GREASE_VALUES else t for t in h.extension_types())
              for h in hellos}
    sets = {frozenset("G" if t in GREASE_VALUES else t for t in h.extension_types())
            for h in hellos}
    greased_lists = sum([
        any(c in GREASE_VALUES for c in hellos[0].cipher_suites),
        any(t in GREASE_VALUES for t in hellos[0].extension_types()),
    ])
    return {
        "extensions": len(hellos[0].extensions),
        "cipher suites": len(hellos[0].cipher_suites),
        "session id bytes": len(hellos[0].session_id),
        "hello bytes": len(hellos[0].raw),
        "distinct orders in %d" % rounds: len(orders),
        "distinct sets in %d" % rounds: len(sets),
        "greases its lists": "yes" if greased_lists else "no",
    }


fox, chrome = shape(firefox157), shape(chrome155)
rows = []
for field in fox:
    a, b = str(fox[field]), str(chrome[field])
    differs = a != b
    rows.append([td(field), td(a, bold=differs), td(b, bold=differs),
                 td("differs" if differs else "", DIM)])
table(["", "Firefox 157", "Chrome 155", ""], rows)

Asked four times each, the orders themselves:

In [ ]:
# The same question asked of each: does the order repeat?
for name, profile in (("firefox157", firefox157), ("chrome155", chrome155)):
    seen = []
    for _ in range(4):
        hello = parse_client_hello(build_client_hello("example.com", profile=profile))
        seen.append(" ".join(
            "G" if t in GREASE_VALUES else str(t) for t in hello.extension_types()))
    display(HTML(
        '<p style="font:13px/1.5 ui-monospace,Menlo,monospace;color:%s">%s</p>'
        % (DIM, name)
        + "".join(
            '<div style="font:12px/1.5 ui-monospace,Menlo,monospace;'
            'white-space:pre;overflow-x:auto">%s</div>' % html.escape(line)
            for line in seen)
    ))

The HTTP/2 side, where the two agree about nothing:

In [ ]:
# The HTTP/2 fingerprint, which matches in no field at all.
fields = [
    ("SETTINGS", lambda p: " ".join("%d:%d" % (k, v) for k, v in p.H2_SETTINGS)),
    ("connection WINDOW_UPDATE", lambda p: str(p.H2_CONNECTION_WINDOW_UPDATE)),
    ("first stream id", lambda p: str(p.H2_FIRST_STREAM_ID)),
    ("HEADERS priority weight", lambda p: str(p.H2_HEADERS_PRIORITY["weight"])),
    ("exclusive", lambda p: str(p.H2_HEADERS_PRIORITY["exclusive"])),
    ("pseudo-header order", lambda p: " ".join(p.H2_PSEUDO_ORDER)),
]
rows = []
for label, read in fields:
    a, b = read(firefox157), read(chrome155)
    rows.append([td(label), td(a, bold=True), td(b, bold=True),
                 td("differs" if a != b else "SAME", DIM)])
table(["", "Firefox 157", "Chrome 155", ""], rows)

And the same targets as section 3, with a third arm. The point is not that
Chrome does better -- it is that two profiles measured the same way both work,
which is the only evidence that the thing being built is a profile system
rather than one browser hard-coded twice.

In [ ]:
from pybrws.client import get as pybrws_get


def fetch_as(profile, url):
    try:
        response = pybrws_get(url, profile=profile, follow_redirects=5, timeout=30)
        return response.status
    except Exception:
        # Reported as a failed cell rather than raised: one refusal should
        # not stop the sweep.
        return None


rows = []
for url in TARGETS:
    cells = [td(url.replace("https://", ""))]
    for label, arm in (("httpx", None), ("firefox", firefox157),
                       ("chrome", chrome155)):
        status = fetch("httpx", url)[0] if arm is None else fetch_as(arm, url)
        colour, text = verdict(status)
        cells.append(td(text, colour, bold=True))
    rows.append(cells)
table(["", "httpx", "pybrws + firefox", "pybrws + chrome"], rows)

What writing the Chrome profile actually produced, which is the part worth
saying out loud: three of its hand-typed constants were wrong and the tests
caught all three. One payload had been **invented** rather than extracted --
186 bytes of plausible-looking structure with nothing behind it. One
extension was missing, giving nineteen where the browser sends twenty. And
two greased extension types could collide, which RFC 8446 forbids and which
produced a hello one byte shorter than any capture.

A length no browser produces is the cheapest tell there is. Every one of
those was found by comparing against a capture instead of against what had
been typed, which is the only reason this section is a result rather than a
guess.